# Homework: Markov Chain Monte Carlo

## Problem 1: Random Walk Metropolis-Hastings

Implement a random walk Metropolis-Hastings sampler to draw samples from the following target distribution, which is a mixture of two univariate normals:

$$p(x) \propto 0.3 \cdot \phi(x; -3, 1) + 0.7 \cdot \phi(x; 2, 0.5^2)$$

where $\phi(x; \mu, \sigma^2)$ is the normal density with mean $\mu$ and variance $\sigma^2$.

**(a)** Write a function `log_mixture_density` that computes the log of the unnormalized target density. Use `np.logaddexp` to avoid numerical overflow when adding the two components on the log scale.

In [ ]:
import numpy as np
from scipy import stats

def log_mixture_density(x):
    log_component_1 = np.log(0.3) + stats.norm.logpdf(x, loc=-3, scale=1.0)
    log_component_2 = np.log(0.7) + stats.norm.logpdf(x, loc=2, scale=0.5)
    return np.logaddexp(log_component_1, log_component_2)

For example, `log_mixture_density(2.0)` should return approximately `-0.583`.

**(b)** Write a function `random_walk_mh` that implements the random walk Metropolis-Hastings algorithm. The function should take the log target density, an initial value, a proposal standard deviation, and the number of iterations, and return a dictionary containing the samples and the acceptance rate.

In [ ]:
def random_walk_mh(log_target, x0, proposal_sd, n_iter, rng=None):
    """Random walk Metropolis-Hastings for a univariate target.

    Parameters
    ----------
    log_target : callable
        Log of the target density (up to a constant).
    x0 : float
        Initial value.
    proposal_sd : float
        Standard deviation of the Gaussian proposal.
    n_iter : int
        Number of iterations.
    rng : numpy random Generator, optional
        Random number generator.

    Returns
    -------
    dict with keys:
        samples : np.ndarray of shape (n_iter,)
        acceptance_rate : float
    """
    if rng is None:
        rng = np.random.default_rng()
    if n_iter <= 0:
        raise ValueError("n_iter must be positive")
    if proposal_sd <= 0:
        raise ValueError("proposal_sd must be positive")

    samples = np.empty(n_iter, dtype=float)
    current = float(x0)
    current_log_target = log_target(current)
    samples[0] = current
    n_accept = 0

    for t in range(1, n_iter):
        proposal = rng.normal(loc=current, scale=proposal_sd)
        proposal_log_target = log_target(proposal)
        log_accept_ratio = proposal_log_target - current_log_target

        if np.log(rng.uniform()) < min(0.0, log_accept_ratio):
            current = proposal
            current_log_target = proposal_log_target
            n_accept += 1

        samples[t] = current

    acceptance_rate = n_accept / max(n_iter - 1, 1)
    return {"samples": samples, "acceptance_rate": acceptance_rate}

**(c)** Run the sampler for 20,000 iterations with `proposal_sd=1.5` starting from `x0=0.0` using `np.random.default_rng(42)`. Report the acceptance rate, the posterior mean (discarding the first 5,000 samples as burn-in), and plot a histogram of the post-burn-in samples overlaid with the true density.

In [ ]:
import matplotlib.pyplot as plt

rw_mh_result = random_walk_mh(
    log_mixture_density,
    x0=0.0,
    proposal_sd=1.5,
    n_iter=20_000,
    rng=np.random.default_rng(42),
)

burn_in = 5_000
post_burn_in = rw_mh_result["samples"][burn_in:]
posterior_mean = post_burn_in.mean()

print(f"Acceptance rate: {rw_mh_result['acceptance_rate']:.3f}")
print(f"Posterior mean after burn-in: {posterior_mean:.3f}")

x_grid = np.linspace(post_burn_in.min() - 1.0, post_burn_in.max() + 1.0, 400)
true_density = (
    0.3 * stats.norm.pdf(x_grid, loc=-3, scale=1.0)
    + 0.7 * stats.norm.pdf(x_grid, loc=2, scale=0.5)
)

plt.figure(figsize=(8, 4))
plt.hist(post_burn_in, bins=50, density=True, alpha=0.6, label="MH samples")
plt.plot(x_grid, true_density, color="black", linewidth=2, label="True density")
plt.xlabel("x")
plt.ylabel("Density")
plt.title("Random Walk Metropolis-Hastings")
plt.legend()
plt.show()


## Problem 2: Diagnosing a Broken Gibbs Sampler

A colleague implemented a Gibbs sampler for a bivariate normal distribution $(\theta_1, \theta_2)$ with mean $(0, 0)$, variances $(1, 1)$, and correlation $\rho$. The full conditionals are $\theta_1 | \theta_2 \sim N(\rho \theta_2, 1 - \rho^2)$ and $\theta_2 | \theta_1 \sim N(\rho \theta_1, 1 - \rho^2)$. Here is their code:

In [7]:
import numpy as np

def gibbs_bivariate(rho, n_iter, rng=None):
    if rng is None:
        rng = np.random.default_rng()
    samples = np.zeros((n_iter, 2))
    theta1, theta2 = 0.0, 0.0
    for t in range(n_iter):
        theta1 = rng.normal(rho * theta2, np.sqrt(1 - rho))
        theta2 = rng.normal(rho * theta1, np.sqrt(1 - rho))
        samples[t] = [theta1, theta2]
    return samples

**(a)** The code has a bug in the conditional standard deviation. Identify the error and explain what the correct expression should be. Describe what effect this bug would have on the distribution of the samples.

**Answer:** The code uses `np.sqrt(1 - rho)` as the conditional standard deviation, but the full conditional variance is $1 - \rho^2$, so the correct standard deviation is `np.sqrt(1 - rho**2)`. Using `np.sqrt(1 - rho)` makes each conditional update too diffuse when $0 < \rho < 1$ because $1 - \rho > 1 - \rho^2$. As a result, the chain has inflated marginal variances and does not reproduce the intended bivariate normal target.

**(b)** Suppose the colleague "fixes" the code by rewriting the update for $\theta_2$ as follows, while leaving $\theta_1$'s update unchanged:

In [9]:
# theta2 = rng.normal(rho * old_theta1, np.sqrt(1 - rho**2))

where `old_theta1` is the value of $\theta_1$ from the *previous* iteration (before the current update). This is no longer a standard Gibbs sampler. Explain why using the stale value of $\theta_1$ changes the algorithm's behavior. Does the modified sampler still target the correct bivariate normal distribution?

**Answer:** This is no longer a standard Gibbs sampler, because a true Gibbs update for $\theta_2$ must condition on the newly updated current value of $\theta_1$. Using the stale value breaks the exact full-conditional update sequence and changes the transition kernel. In general, that modified chain does **not** target the correct bivariate normal distribution, because the proper Gibbs invariance argument no longer applies.

**(c)** Write a corrected version of `gibbs_bivariate`. Run it with $\rho = 0.8$ for 10,000 iterations using `np.random.default_rng(0)`. Verify correctness by computing the sample mean, sample variances, and sample correlation of the post-burn-in samples (discarding the first 1,000).




In [10]:
def gibbs_bivariate_corrected(rho, n_iter, rng=None):
    if rng is None:
        rng = np.random.default_rng()
    samples = np.zeros((n_iter, 2))
    theta1, theta2 = 0.0, 0.0
    conditional_sd = np.sqrt(1 - rho**2)

    for t in range(n_iter):
        theta1 = rng.normal(rho * theta2, conditional_sd)
        theta2 = rng.normal(rho * theta1, conditional_sd)
        samples[t] = [theta1, theta2]

    return samples

samples = gibbs_bivariate_corrected(rho=0.8, n_iter=10_000, rng=np.random.default_rng(0))
post_burn_in = samples[1_000:]
problem2_summary = {
    "sample_mean": post_burn_in.mean(axis=0),
    "sample_variances": post_burn_in.var(axis=0),
    "sample_correlation": np.corrcoef(post_burn_in.T)[0, 1],
}

print("Sample mean:", np.round(problem2_summary["sample_mean"], 3))
print("Sample variances:", np.round(problem2_summary["sample_variances"], 3))
print(f"Sample correlation: {problem2_summary['sample_correlation']:.3f}")

Sample mean: [0.025 0.025]
Sample variances: [1.01  1.002]
Sample correlation: 0.801


## Problem 3: Convergence Diagnostics

**(a)** Write a function `compute_ess` that estimates the effective sample size of an MCMC chain. Use the autocorrelation approach: compute autocorrelations via FFT and sum consecutive pairs of autocorrelations, stopping at the first pair whose sum is negative (Geyer's initial positive sequence method). The ESS is $T / (1 + 2\sum_k \hat{\rho}(k))$ where the sum uses the truncated autocorrelation series.

In [14]:
def compute_ess(chain):
    """Compute effective sample size using Geyer's initial positive sequence.

    Parameters
    ----------
    chain : np.ndarray of shape (T,)
        MCMC samples.

    Returns
    -------
    float : Effective sample size.
    """
    chain = np.asarray(chain, dtype=float)
    n = chain.size
    if n < 2:
        return float(n)

    centered = chain - chain.mean()
    variance = np.dot(centered, centered) / n
    if variance == 0:
        return float(n)

    fft_size = 1 << (2 * n - 1).bit_length()
    fft_vals = np.fft.rfft(centered, n=fft_size)
    acov = np.fft.irfft(fft_vals * np.conjugate(fft_vals), n=fft_size)[:n]
    acov /= np.arange(n, 0, -1)
    autocorr = acov / acov[0]

    positive_autocorr_sum = 0.0
    for lag in range(1, n - 1, 2):
        pair_sum = autocorr[lag] + autocorr[lag + 1]
        if pair_sum < 0:
            break
        positive_autocorr_sum += pair_sum

    if (n - 1) % 2 == 1 and n > 1:
        last_lag = n - 1
        if autocorr[last_lag] > 0:
            positive_autocorr_sum += autocorr[last_lag]

    ess = n / (1 + 2 * positive_autocorr_sum)
    return float(min(n, max(1.0, ess)))

For example, for a chain of 1,000 independent standard normal draws, `compute_ess` should return a value close to 1,000. For a highly autocorrelated chain (e.g., a random walk with small steps), the ESS should be much smaller than the chain length.


In [ ]:
rng = np.random.default_rng(0)
iid_chain = rng.normal(size=1_000)

autocorrelated_chain = np.empty(1_000)
autocorrelated_chain[0] = rng.normal()
for t in range(1, 1_000):
    autocorrelated_chain[t] = autocorrelated_chain[t - 1] + rng.normal(scale=0.1)

print(f"ESS for 1,000 iid normal draws: {compute_ess(iid_chain):.1f}")
print(f"ESS for highly autocorrelated chain: {compute_ess(autocorrelated_chain):.1f}")

ESS for 1,000 iid normal draws: 1000.0
ESS for highly autocorrelated chain: 4.4



**(b)** Write a function `gelman_rubin` that computes the $\hat{R}$ statistic given multiple MCMC chains for the same parameter. The function should accept a list of 1D arrays (one per chain) and return the $\hat{R}$ value.

In [15]:
def gelman_rubin(chains):
    """Compute the Gelman-Rubin R-hat statistic.

    Parameters
    ----------
    chains : list of np.ndarray
        Each array is a 1D chain of samples for the same parameter.

    Returns
    -------
    float : R-hat statistic.
    """
    chains = [np.asarray(chain, dtype=float) for chain in chains]
    if len(chains) < 2:
        raise ValueError("gelman_rubin requires at least two chains")

    lengths = {chain.size for chain in chains}
    if len(lengths) != 1:
        raise ValueError("all chains must have the same length")

    n = chains[0].size
    if n < 2:
        raise ValueError("each chain must contain at least two draws")

    m = len(chains)
    chain_means = np.array([chain.mean() for chain in chains])
    chain_vars = np.array([chain.var(ddof=1) for chain in chains])

    w = chain_vars.mean()
    b = n * chain_means.var(ddof=1)
    var_hat = ((n - 1) / n) * w + b / n
    return float(np.sqrt(var_hat / w))

For example, if all chains are drawn from the same distribution and are long enough, `gelman_rubin` should return a value close to 1.0.


In [ ]:
rng = np.random.default_rng(1)
example_chains = [rng.normal(size=2_000) for _ in range(4)]

print(f"R-hat for four similar chains: {gelman_rubin(example_chains):.4f}")

R-hat for four similar chains: 0.9998



**(c)** Using the `random_walk_mh` function from Problem 1 and the mixture target, run 4 chains of 15,000 iterations each from starting values $x_0 \in \{-5, -1, 3, 7\}$ with `proposal_sd=1.5`. Use seeds `np.random.default_rng(i)` for chain $i = 0, 1, 2, 3$. Discard 5,000 as burn-in. Report the ESS for each chain and the $\hat{R}$ statistic.

In [16]:
initial_values = [-5, -1, 3, 7]
chains = []
ess_per_chain = []

for i, x0 in enumerate(initial_values):
    result = random_walk_mh(
        log_mixture_density,
        x0=x0,
        proposal_sd=1.5,
        n_iter=15_000,
        rng=np.random.default_rng(i),
    )
    post_burn_in_chain = result["samples"][5_000:]
    chains.append(post_burn_in_chain)
    ess_per_chain.append(compute_ess(post_burn_in_chain))

rhat = gelman_rubin(chains)
problem3_summary = {
    "ess_per_chain": ess_per_chain,
    "rhat": rhat,
}

for i, ess in enumerate(ess_per_chain):
    print(f"Chain {i} ESS: {ess:.1f}")
print(f"R-hat: {rhat:.4f}")

Chain 0 ESS: 106.2
Chain 1 ESS: 150.0
Chain 2 ESS: 87.1
Chain 3 ESS: 138.0
R-hat: 1.0040


## Problem 4: Gibbs Sampler for Bayesian Normal Model

Consider the Bayesian model where we observe $y_1, \ldots, y_n$ from a normal distribution with unknown mean and variance:

$$y_i | \mu, \sigma^2 \sim N(\mu, \sigma^2), \quad \mu \sim N(\mu_0, \sigma_0^2), \quad \sigma^2 \sim \text{Inv-Gamma}(a_0, b_0)$$

The full conditional distributions are:

$$\mu | \sigma^2, \mathbf{y} \sim N\left(\frac{\sigma_0^{-2} \mu_0 + n \sigma^{-2} \bar{y}}{\sigma_0^{-2} + n \sigma^{-2}}, \, \frac{1}{\sigma_0^{-2} + n \sigma^{-2}}\right)$$

$$\sigma^2 | \mu, \mathbf{y} \sim \text{Inv-Gamma}\left(a_0 + \frac{n}{2}, \, b_0 + \frac{1}{2}\sum_{i=1}^n (y_i - \mu)^2\right)$$

**(a)** Implement the Gibbs sampler for this model.

In [ ]:
def gibbs_normal(y, mu0=0.0, sigma0=10.0, a0=0.01, b0=0.01,
                 n_iter=10000, rng=None):
    """Gibbs sampler for Bayesian normal model.

    Parameters
    ----------
    y : np.ndarray of shape (n,)
        Observed data.
    mu0 : float
        Prior mean for mu.
    sigma0 : float
        Prior standard deviation for mu.
    a0, b0 : float
        Inverse-Gamma prior parameters for sigma^2.
    n_iter : int
        Number of iterations.
    rng : numpy random Generator, optional
        Random number generator.

    Returns
    -------
    dict with keys:
        mu_samples : np.ndarray of shape (n_iter,)
        sigma2_samples : np.ndarray of shape (n_iter,)
    """
    # Your code here
    pass

**(b)** Generate data with `np.random.default_rng(42)`: $n = 50$ observations from $N(\mu=3, \sigma^2=4)$. Run the Gibbs sampler for 10,000 iterations using `np.random.default_rng(0)` with the default prior settings. Discard the first 2,000 samples as burn-in. Report the posterior means and 95% credible intervals for $\mu$ and $\sigma^2$.

**(c)** Produce trace plots for both $\mu$ and $\sigma^2$. Compute the ESS for each parameter using your function from Problem 3. Comment on the mixing quality of the sampler.

## Problem 5: Independence Metropolis-Hastings

The random walk Metropolis-Hastings algorithm proposes new states relative to the current state. An alternative is the **independence sampler**, where the proposal $q(x^*)$ does not depend on the current state $x_t$. The acceptance ratio becomes:

$$\alpha(x_t, x^*) = \min\left(1, \frac{\pi(x^*) \, q(x_t)}{\pi(x_t) \, q(x^*)}\right)$$

where $\pi$ is the target density and $q$ is the proposal density, both evaluable up to normalizing constants.

**(a)** Implement an independence Metropolis-Hastings sampler.

In [ ]:
def independence_mh(log_target, log_proposal, proposal_sampler,
                    x0, n_iter, rng=None):
    """Independence Metropolis-Hastings sampler.

    Parameters
    ----------
    log_target : callable
        Log of the target density (up to a constant), x -> float.
    log_proposal : callable
        Log of the proposal density, x -> float.
    proposal_sampler : callable
        Function that draws a sample from the proposal, rng -> float.
    x0 : float
        Initial value.
    n_iter : int
        Number of iterations.
    rng : numpy random Generator, optional
        Random number generator.

    Returns
    -------
    dict with keys:
        samples : np.ndarray of shape (n_iter,)
        acceptance_rate : float
    """
    # Your code here
    pass

**(b)** Consider the same mixture-of-normals target from Problem 1. Use a $t$-distribution with 5 degrees of freedom, location 1.0, and scale 3.0 as the proposal distribution. Run the independence sampler for 20,000 iterations starting from `x0=0.0` with `np.random.default_rng(42)`. Report the acceptance rate and compare the ESS (post-burn-in, discarding first 5,000) to the random walk MH result from Problem 1(c).

**(c)** Now use a $N(0, 0.5^2)$ proposal instead. Run the sampler with the same settings and report the acceptance rate and ESS. Explain why the performance differs so dramatically from part (b). In your explanation, discuss what property the proposal distribution must have relative to the target for the independence sampler to work well, and why a poorly chosen proposal can be worse than the random walk approach.